# Step 7: RQ3, do cities fall into distinct pollution profiles?

Unit of analysis: the **city**. Each city is described by its typical pollution level, its
extremes, its share of days per AQI band and its seasonality. Then PCA (and the same via SVD)
finds the main axes of variation, and k-means and hierarchical clustering group the cities.

City selection: every city with at least one full year of data (first reading on or before
2019-07-01), so that each profile covers all seasons. This keeps 19 cities and leaves out
Shillong, Chandigarh, Bhopal, Ernakulam, Kochi and Aizawl (and Mumbai, dropped in Step 2).
March-May 2020 (lockdown) is excluded from the profiles: it would lower the averages of short-
history cities much more than of long-history cities.

No train/test split is needed: this is unsupervised description, not prediction.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:                                   # Colab: get the repo (code + data) first
    import google.colab  # noqa: F401
    if not (ROOT / "src" / "aq.py").exists():
        import subprocess
        subprocess.run(["git", "clone", "-b", "claude/wizardly-meitner-nm2sfq",
                        "https://github.com/s-aura-bhhh/Air-quality-analysis.git"], check=True)
        ROOT = Path.cwd() / "Air-quality-analysis"
except ImportError:
    pass
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import aq

FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"07_{name}.png", dpi=150); plt.show()

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

In [2]:
df = aq.load_checked()
first = df.groupby("City")["Date"].min()
CITIES = sorted(first[first <= "2019-07-01"].index)
d = df[df["City"].isin(CITIES) & ~aq.in_lockdown(df["Date"])].copy()
print(len(CITIES), "cities:", CITIES)

# Pollutants usable for every city: observed on at least 25% of days in each city.
# (25%, not 30%: Jorapokhar has PM2.5 on 27% of days, and PM2.5 is the key pollutant.)
cover = d.groupby("City")[aq.POLLUTANTS].apply(lambda g: g.notna().mean())
POLS = [p for p in aq.POLLUTANTS if cover[p].min() >= 0.25]
print("pollutants used:", POLS)
print("left out (a city lacks them):", [p for p in aq.POLLUTANTS if p not in POLS])
cover[aq.POLLUTANTS].round(2).min().to_frame("lowest city coverage").T

19 cities: ['Ahmedabad', 'Amaravati', 'Amritsar', 'Bengaluru', 'Brajrajnagar', 'Chennai', 'Coimbatore', 'Delhi', 'Gurugram', 'Guwahati', 'Hyderabad', 'Jaipur', 'Jorapokhar', 'Kolkata', 'Lucknow', 'Patna', 'Talcher', 'Thiruvananthapuram', 'Visakhapatnam']
pollutants used: ['PM2.5', 'NO', 'NO2', 'CO', 'SO2', 'O3']
left out (a city lacks them): ['PM10', 'NOx', 'NH3', 'Benzene', 'Toluene']


,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene
lowest city coverage,0.27,0.0,0.68,0.68,0.0,0.0,0.54,0.67,0.64,0.0,0.0


## 7.1 City feature matrix

For each city and each usable pollutant: the **median** (typical day) and the **90th percentile**
(bad days), on a log1p scale so that one extreme city does not dominate the standardised matrix.
Plus the share of days Good/Satisfactory, Moderate and Poor-or-worse, and the
**winter-to-monsoon PM2.5 ratio** (median Nov-Feb / median Jun-Sep). Monsoon is used instead of
summer because for two cities the only spring data falls in the excluded lockdown.

In [3]:
WINTER, MONSOON = [11, 12, 1, 2], [6, 7, 8, 9]

def city_features(d):
    g = d.groupby("City")
    f = pd.concat([np.log1p(g[POLS].median()).add_prefix("med_"),
                   np.log1p(g[POLS].quantile(0.9)).add_prefix("p90_")], axis=1)
    b = d.dropna(subset=["AQI"])
    band = pd.cut(b["AQI"], [-np.inf, 100, 200, np.inf], labels=["share_good_sat", "share_moderate", "share_poor_plus"])
    f = f.join(pd.crosstab(b["City"], band, normalize="index"))
    m = d["Date"].dt.month
    w = d[m.isin(WINTER)].groupby("City")["PM2.5"].median()
    s = d[m.isin(MONSOON)].groupby("City")["PM2.5"].median()
    f["log_winter_monsoon_pm25"] = np.log(w / s)
    return f

feat = city_features(d)
print(feat.shape, "missing cells:", int(feat.isna().sum().sum()))
feat.round(2).to_csv(TAB / "07_city_features.csv")
feat.round(2)

(19, 16) missing cells: 0


,med_PM2.5,med_NO,med_NO2,med_CO,med_SO2,med_O3,p90_PM2.5,p90_NO,p90_NO2,p90_CO,p90_SO2,p90_O3,share_good_sat,share_moderate,share_poor_plus,log_winter_monsoon_pm25
City,,,,,,,,,,,,,,,,
Ahmedabad,4.13,2.90,3.94,2.90,3.90,3.67,4.77,3.90,4.86,3.90,4.78,4.23,0.03,0.11,0.86,0.68
Amaravati,3.47,1.53,2.87,0.48,2.63,3.56,4.41,2.27,4.03,0.73,3.14,4.14,0.64,0.29,0.07,1.33
Amritsar,3.86,2.71,2.70,0.43,1.97,3.05,4.60,3.50,3.59,0.76,2.73,3.58,0.46,0.42,0.12,0.50
Bengaluru,3.48,2.18,3.35,0.66,1.78,3.41,4.12,2.89,3.80,1.29,2.16,4.07,0.63,0.34,0.02,0.71
Brajrajnagar,4.12,2.63,2.85,1.09,2.53,2.37,4.74,3.25,3.33,1.52,2.83,3.39,0.18,0.59,0.23,1.01
Chennai,3.87,2.21,2.82,0.62,2.03,3.43,4.41,2.71,3.26,1.06,2.71,3.98,0.48,0.45,0.07,0.24
Coimbatore,3.43,2.24,2.85,0.70,2.35,3.21,3.84,2.55,3.85,0.99,2.60,3.89,0.96,0.04,0.00,0.47
Delhi,4.59,3.39,3.90,0.82,2.74,3.81,5.45,4.48,4.41,1.43,3.25,4.48,0.08,0.24,0.68,1.26
Gurugram,4.55,2.36,3.02,0.55,2.08,3.43,5.41,3.74,3.82,1.09,2.93,4.01,0.17,0.28,0.55,0.99


In [4]:
Xs = StandardScaler().fit_transform(feat.values)
Xs = pd.DataFrame(Xs, index=feat.index, columns=feat.columns)

## 7.2 PCA, and the same result from the SVD

For the centred (here also standardised) matrix X (n cities × p features), the SVD is
X = U S Vᵀ. The columns of V are the principal directions (loadings), the PC scores are XV = US,
and the variance explained by component j is s_j² / (n − 1). The cell below computes both ways and
checks that they agree (loadings agree up to sign, which is arbitrary).

In [5]:
pca = PCA().fit(Xs)
U, s, Vt = np.linalg.svd(Xs.values - Xs.values.mean(axis=0), full_matrices=False)
ev_svd = s ** 2 / (len(Xs) - 1)
print("eigenvalues equal:", np.allclose(pca.explained_variance_, ev_svd))
print("loadings equal up to sign:", np.allclose(np.abs(pca.components_), np.abs(Vt)))
print("scores equal up to sign:", np.allclose(np.abs(pca.transform(Xs)), np.abs(U * s)))
evr = pca.explained_variance_ratio_
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(evr) + 1), evr, label="per component")
ax.plot(range(1, len(evr) + 1), np.cumsum(evr), marker="o", color="C1", label="cumulative")
ax.axhline(0.8, color="grey", ls="--", lw=0.8); ax.set_xlabel("component"); ax.legend()
ax.set_title("PCA of city profiles: explained variance")
save("scree")
N_PC = int(np.searchsorted(np.cumsum(evr), 0.8) + 1)
print("components for 80% of variance:", N_PC, "| first three:", evr[:3].round(3))

eigenvalues equal: True
loadings equal up to sign: True
scores equal up to sign: True


components for 80% of variance: 4 | first three: [0.423 0.178 0.136]


In [6]:
load = pd.DataFrame(pca.components_[:3].T, index=feat.columns, columns=["PC1", "PC2", "PC3"])
# sign convention: make PC1 positive for 'more pollution'
for c in load:
    if load.loc["med_PM2.5", c] < 0 and c == "PC1":
        load[c] *= -1
load.round(3).to_csv(TAB / "07_loadings.csv")
fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(load, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Loadings of the first three components")
save("loadings")

**For the report:** name each component from its largest loadings. Typical reading: PC1 = overall
pollution level (all medians, p90s and the Poor-or-worse share load together); PC2 = a contrast,
e.g. particulate/seasonal (PM, winter ratio) versus gaseous or industrial (SO2, CO, NO2); PC3 =
whatever is left (often one city-specific pollutant). Check the actual signs before naming.

In [7]:
scores = pd.DataFrame(pca.transform(Xs)[:, :max(N_PC, 3)], index=feat.index,
                      columns=[f"PC{i + 1}" for i in range(max(N_PC, 3))])
if pca.components_[0][list(feat.columns).index("med_PM2.5")] < 0:
    scores["PC1"] *= -1
sign1 = 1 if pca.components_[0][list(feat.columns).index("med_PM2.5")] >= 0 else -1
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(scores["PC1"], scores["PC2"], s=40)
for city, r in scores.iterrows():
    ax.annotate(city, (r["PC1"], r["PC2"]), fontsize=9, xytext=(3, 3), textcoords="offset points")
scale = np.abs(scores[["PC1", "PC2"]]).max().max() / np.abs(load[["PC1", "PC2"]]).max().max() * 0.8
top = load[["PC1", "PC2"]].pow(2).sum(axis=1).sort_values(ascending=False).head(10).index
for f_ in top:
    ax.arrow(0, 0, load.loc[f_, "PC1"] * scale, load.loc[f_, "PC2"] * scale, color="#DD8452", alpha=0.7, head_width=0.08)
    ax.text(load.loc[f_, "PC1"] * scale * 1.08, load.loc[f_, "PC2"] * scale * 1.08, f_, color="#AA5522", fontsize=8)
ax.axhline(0, color="grey", lw=0.6); ax.axvline(0, color="grey", lw=0.6)
ax.set_xlabel(f"PC1 ({evr[0]:.0%})"); ax.set_ylabel(f"PC2 ({evr[1]:.0%})")
ax.set_title("Biplot: cities and the 10 strongest feature directions")
save("biplot")

## 7.3 Clustering

k-means on the PC scores that cover 80% of the variance (dropping the noisiest components).
k is chosen with the elbow (inertia) and the silhouette score; hierarchical clustering
(Ward linkage) is the second method.

In [8]:
P = scores.iloc[:, :N_PC].values
ks = range(2, 8)
km = {k: KMeans(k, n_init=50, random_state=aq.SEED).fit(P) for k in ks}
inertia = [km[k].inertia_ for k in ks]
sil = [silhouette_score(P, km[k].labels_) for k in ks]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(ks, inertia, marker="o"); axes[0].set(xlabel="k", ylabel="within-cluster SS", title="Elbow")
axes[1].plot(ks, sil, marker="o", color="C1"); axes[1].set(xlabel="k", ylabel="silhouette", title="Silhouette")
save("choose_k")
K = list(ks)[int(np.argmax(sil))]
print("silhouette by k:", dict(zip(ks, np.round(sil, 3))), "-> k =", K)
labels_km = km[K].labels_

silhouette by k: {2: np.float64(0.313), 3: np.float64(0.296), 4: np.float64(0.379), 5: np.float64(0.374), 6: np.float64(0.315), 7: np.float64(0.305)} -> k = 4


In [9]:
Zl = linkage(P, method="ward")
fig, ax = plt.subplots(figsize=(11, 4.5))
dendrogram(Zl, labels=list(feat.index), leaf_rotation=60, ax=ax)
ax.set_title("Hierarchical clustering (Ward) on the same PC scores")
save("dendrogram")
labels_h = fcluster(Zl, K, criterion="maxclust")
print("agreement k-means vs hierarchical, adjusted Rand index:", round(adjusted_rand_score(labels_km, labels_h), 3))
pd.crosstab(pd.Series(labels_km, name="k-means"), pd.Series(labels_h, name="hierarchical"))

agreement k-means vs hierarchical, adjusted Rand index: 0.831


hierarchical,1,2,3,4
k-means,,,,
0,5,0,0,0
1,0,0,8,1
2,0,1,0,0
3,0,0,0,4


ARI = 1 means identical partitions, 0 means no more agreement than chance.

## 7.4 Profile and name the clusters

In [10]:
REGION = {"Ahmedabad": "West", "Amaravati": "South", "Amritsar": "North", "Bengaluru": "South",
          "Brajrajnagar": "East", "Chennai": "South", "Coimbatore": "South", "Delhi": "North",
          "Gurugram": "North", "Guwahati": "Northeast", "Hyderabad": "South", "Jaipur": "North",
          "Jorapokhar": "East", "Kolkata": "East", "Lucknow": "North", "Patna": "East",
          "Talcher": "East", "Thiruvananthapuram": "South", "Visakhapatnam": "South"}
IGP = {"Amritsar", "Delhi", "Gurugram", "Lucknow", "Patna", "Kolkata"}   # Indo-Gangetic Plain
cl = pd.DataFrame({"cluster": labels_km, "hierarchical": labels_h}, index=feat.index)
cl["region"] = cl.index.map(REGION)
cl["Indo_Gangetic_Plain"] = cl.index.isin(IGP)
cl = cl.join(scores[["PC1", "PC2"]].round(2)).sort_values(["cluster", "PC1"])
cl.to_csv(TAB / "07_city_clusters.csv")
cl

,cluster,hierarchical,region,Indo_Gangetic_Plain,PC1,PC2
City,,,,,,
Kolkata,0,1,East,True,0.62,0.37
Gurugram,0,1,North,True,1.32,-0.23
Lucknow,0,1,North,True,1.80,0.47
Patna,0,1,East,True,3.65,-0.37
Delhi,0,1,North,True,4.13,1.17
Thiruvananthapuram,1,3,South,False,-4.65,1.08
Coimbatore,1,3,South,False,-3.07,1.47
Bengaluru,1,3,South,False,-2.39,1.65
Amaravati,1,3,South,False,-2.20,1.04


In [11]:
raw_prof = d.groupby("City")[POLS].median().add_prefix("median_")
raw_prof["PM2.5_winter/monsoon"] = np.exp(feat["log_winter_monsoon_pm25"])
raw_prof = raw_prof.join(feat[["share_good_sat", "share_moderate", "share_poor_plus"]])
prof = raw_prof.groupby(cl["cluster"]).mean()
prof.insert(0, "n_cities", cl.groupby("cluster").size())
prof.insert(1, "cities", cl.groupby("cluster").apply(lambda g: ", ".join(g.index), include_groups=False))
prof.round(2).to_csv(TAB / "07_cluster_profiles.csv")
prof.round(2).T

cluster,0,1,2,3
n_cities,5,9,1,4
cities,"Kolkata, Gurugram, Lucknow, Patna, Delhi","Thiruvananthapuram, Coimbatore, Bengaluru, Ama...",Ahmedabad,"Guwahati, Jorapokhar, Brajrajnagar, Talcher"
median_PM2.5,85.71,38.48,61.12,54.89
median_NO,15.72,8.1,17.19,14.27
median_NO2,32.14,21.11,50.61,11.6
median_CO,1.12,0.78,17.16,1.19
median_SO2,10.0,8.18,48.49,19.34
median_O3,32.16,30.71,38.44,15.88
PM2.5_winter/monsoon,4.02,2.23,1.98,4.22
share_good_sat,0.21,0.57,0.03,0.28


In [12]:
fig, ax = plt.subplots(figsize=(10, 8))
for k_, g in cl.groupby("cluster"):
    ax.scatter(g["PC1"], g["PC2"], s=60, label=f"cluster {k_}")
    for city, r in g.iterrows():
        ax.annotate(city + (" *" if city in IGP else ""), (r["PC1"], r["PC2"]), fontsize=9,
                    xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()
ax.set_title("City clusters on the first two components (* = Indo-Gangetic Plain)")
save("clusters")
print(pd.crosstab(cl["cluster"], cl["region"]))
print(pd.crosstab(cl["cluster"], cl["Indo_Gangetic_Plain"]))

region   East  North  Northeast  South  West
cluster                                     
0           2      3          0      0     0
1           0      2          0      7     0
2           0      0          0      0     1
3           3      0          1      0     0
Indo_Gangetic_Plain  False  True 
cluster                          
0                        0      5
1                        8      1
2                        1      0
3                        4      0


**For the report:** give each cluster a name from its profile row (e.g. "high-pollution
Indo-Gangetic cities with strong winter peaks", "cleaner southern and coastal cities",
"industrial eastern towns with high SO2/CO"), check against geography using the two crosstabs,
and name the cities that do not fit their region (and why that might be).

## 7.5 Stability

With 19 cities, clusters can change with small perturbations. Three checks:
1. different k (k-1, k+1);
2. bootstrap: resample each city's days with replacement, rebuild the feature matrix, redo PCA
   and k-means; record how often each pair of cities ends up in the same cluster;
3. drop Ahmedabad (team decision to keep it; its CO readings are suspect).

In [13]:
for k_ in [K - 1, K + 1]:
    if k_ in km:
        print(f"k = {k_}: ARI vs k = {K}:", round(adjusted_rand_score(labels_km, km[k_].labels_), 3))
        print("   ", {c: int(l) for c, l in zip(feat.index, km[k_].labels_)})

k = 3: ARI vs k = 4: 0.751
    {'Ahmedabad': 2, 'Amaravati': 0, 'Amritsar': 0, 'Bengaluru': 0, 'Brajrajnagar': 1, 'Chennai': 0, 'Coimbatore': 0, 'Delhi': 1, 'Gurugram': 1, 'Guwahati': 1, 'Hyderabad': 0, 'Jaipur': 0, 'Jorapokhar': 1, 'Kolkata': 1, 'Lucknow': 1, 'Patna': 1, 'Talcher': 1, 'Thiruvananthapuram': 0, 'Visakhapatnam': 0}
k = 5: ARI vs k = 4: 0.798
    {'Ahmedabad': 3, 'Amaravati': 0, 'Amritsar': 2, 'Bengaluru': 0, 'Brajrajnagar': 2, 'Chennai': 0, 'Coimbatore': 0, 'Delhi': 1, 'Gurugram': 1, 'Guwahati': 2, 'Hyderabad': 0, 'Jaipur': 0, 'Jorapokhar': 4, 'Kolkata': 1, 'Lucknow': 1, 'Patna': 1, 'Talcher': 2, 'Thiruvananthapuram': 0, 'Visakhapatnam': 0}


In [14]:
def cluster_pipeline(feat_, k, n_pc=N_PC):
    Xb = StandardScaler().fit_transform(feat_.values)
    Pb = PCA(n_pc).fit_transform(Xb)
    return KMeans(k, n_init=20, random_state=aq.SEED).fit_predict(Pb)

rng = np.random.default_rng(aq.SEED)
B = 50
co = np.zeros((len(feat), len(feat))); aris = []
groups = [g for _, g in d.groupby("City")]
for b in range(B):
    boot = pd.concat([g.sample(len(g), replace=True, random_state=int(rng.integers(1e9))) for g in groups]).reset_index(drop=True)
    fb = city_features(boot).reindex(feat.index)
    lb = cluster_pipeline(fb, K)
    co += (lb[:, None] == lb[None, :])
    aris.append(adjusted_rand_score(labels_km, lb))
co /= B
print(f"bootstrap ({B} resamples): mean ARI vs original = {np.mean(aris):.3f} (min {np.min(aris):.3f})")
order = cl.index
co_df = pd.DataFrame(co, index=feat.index, columns=feat.index).loc[order, order]
co_df.round(2).to_csv(TAB / "07_bootstrap_coclustering.csv")
fig, ax = plt.subplots(figsize=(10, 8.5))
sns.heatmap(co_df, cmap="viridis", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "share of resamples in same cluster"})
ax.set_title("Bootstrap co-clustering (cities ordered by cluster)")
save("bootstrap")

bootstrap (50 resamples): mean ARI vs original = 0.960 (min 0.514)


In [15]:
keep = feat.index != "Ahmedabad"
lab_wo = cluster_pipeline(feat[keep], K)
print("without Ahmedabad, ARI vs original (other cities):", round(adjusted_rand_score(labels_km[keep], lab_wo), 3))
pd.DataFrame({"with": labels_km[keep], "without": lab_wo}, index=feat.index[keep]).T

without Ahmedabad, ARI vs original (other cities): 0.679


City,Amaravati,Amritsar,Bengaluru,Brajrajnagar,Chennai,Coimbatore,Delhi,Gurugram,Guwahati,Hyderabad,Jaipur,Jorapokhar,Kolkata,Lucknow,Patna,Talcher,Thiruvananthapuram,Visakhapatnam
with,1,1,1,3,1,1,0,0,3,1,1,3,0,0,0,3,1,1
without,1,3,1,2,3,1,0,0,2,3,3,2,0,0,0,2,1,3


**For the report:** cities with co-clustering near 1 with their cluster-mates are robust
assignments; values around 0.5 mean that city sits between clusters. Report the mean bootstrap
ARI and say plainly that with 19 cities the clustering is descriptive, not a definitive
typology.